In [1]:
# Importa las mismas librerias usadas en el notebook de entrenamiento.
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import jupyter
import streamlit as st
import holidays

In [2]:
# Carga los datos de inferencia desde una ruta relativa a notebooks/.
inferencia_df = pd.read_csv(
    "../data/raw/inferencia/ventas_2025_inferencia.csv",
    parse_dates=["fecha"],
)

print(f"Dimensiones de inferencia_df: {inferencia_df.shape}")
inferencia_df.head()

Dimensiones de inferencia_df: (888, 13)


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


In [3]:
# Crea una copia de inferencia_df para conservar intacto el CSV original cargado.
inferencia_df_transformado = inferencia_df.copy()
inferencia_df_transformado["fecha"] = pd.to_datetime(
    inferencia_df_transformado["fecha"]
)

# Construye variables de calendario con las mismas reglas del entrenamiento.
anios_inferencia = sorted(
    inferencia_df_transformado["fecha"].dt.year.dropna().unique().tolist()
)
festivos_espana_inferencia = holidays.country_holidays(
    "ES",
    years=anios_inferencia,
)
inferencia_df_transformado["anio"] = inferencia_df_transformado["fecha"].dt.year
inferencia_df_transformado["mes"] = inferencia_df_transformado["fecha"].dt.month
inferencia_df_transformado["dia_mes"] = inferencia_df_transformado["fecha"].dt.day
inferencia_df_transformado["dia_semana"] = (
    inferencia_df_transformado["fecha"].dt.dayofweek
)
inferencia_df_transformado["dia_del_anio"] = (
    inferencia_df_transformado["fecha"].dt.dayofyear
)
inferencia_df_transformado["semana_anio"] = (
    inferencia_df_transformado["fecha"]
    .dt.isocalendar()
    .week.astype("int64")
)
inferencia_df_transformado["trimestre"] = (
    inferencia_df_transformado["fecha"].dt.quarter
)
inferencia_df_transformado["dias_del_mes"] = (
    inferencia_df_transformado["fecha"].dt.days_in_month
)
inferencia_df_transformado["es_fin_de_semana"] = (
    inferencia_df_transformado["dia_semana"] >= 5
)
inferencia_df_transformado["es_inicio_mes"] = (
    inferencia_df_transformado["dia_mes"] <= 3
)
inferencia_df_transformado["es_fin_mes"] = (
    inferencia_df_transformado["dia_mes"]
    >= inferencia_df_transformado["dias_del_mes"] - 2
)
inferencia_df_transformado["nombre_festivo"] = (
    inferencia_df_transformado["fecha"].dt.date.map(festivos_espana_inferencia)
)
inferencia_df_transformado["es_festivo"] = (
    inferencia_df_transformado["nombre_festivo"].notna()
)

# Marca Black Friday como el viernes posterior al cuarto jueves de noviembre.
primer_dia_noviembre_inferencia = pd.to_datetime(
    inferencia_df_transformado["anio"].astype(str) + "-11-01"
)
dias_hasta_primer_jueves_inferencia = (
    3 - primer_dia_noviembre_inferencia.dt.dayofweek
) % 7
fechas_black_friday_inferencia = (
    primer_dia_noviembre_inferencia
    + pd.to_timedelta(22 + dias_hasta_primer_jueves_inferencia, unit="D")
)
fechas_cyber_monday_inferencia = (
    fechas_black_friday_inferencia + pd.Timedelta(days=3)
)
inferencia_df_transformado["es_black_friday"] = (
    inferencia_df_transformado["fecha"].eq(fechas_black_friday_inferencia)
)
inferencia_df_transformado["es_cyber_monday"] = (
    inferencia_df_transformado["fecha"].eq(fechas_cyber_monday_inferencia)
)
inferencia_df_transformado["es_temporada_navidad"] = (
    (
        inferencia_df_transformado["mes"].eq(12)
        & inferencia_df_transformado["dia_mes"].ge(15)
    )
    | (
        inferencia_df_transformado["mes"].eq(1)
        & inferencia_df_transformado["dia_mes"].le(6)
    )
)
inferencia_df_transformado["dia_semana_sin"] = np.sin(
    2 * np.pi * inferencia_df_transformado["dia_semana"] / 7
)
inferencia_df_transformado["dia_semana_cos"] = np.cos(
    2 * np.pi * inferencia_df_transformado["dia_semana"] / 7
)
inferencia_df_transformado["mes_sin"] = np.sin(
    2 * np.pi * (inferencia_df_transformado["mes"] - 1) / 12
)
inferencia_df_transformado["mes_cos"] = np.cos(
    2 * np.pi * (inferencia_df_transformado["mes"] - 1) / 12
)

# Ordena cada producto en 2025 antes de calcular lags y medias moviles.
inferencia_df_transformado = inferencia_df_transformado.sort_values(
    ["producto_id", "anio", "fecha"]
).reset_index(drop=True)
grupo_unidades_inferencia = inferencia_df_transformado.groupby(
    ["producto_id", "anio"],
    sort=False,
)["unidades_vendidas"]
columnas_lags_inferencia = []
for numero_lag_inferencia in range(1, 8):
    nombre_lag_inferencia = f"unidades_vendidas_lag_{numero_lag_inferencia}"
    inferencia_df_transformado[nombre_lag_inferencia] = (
        grupo_unidades_inferencia.shift(numero_lag_inferencia)
    )
    columnas_lags_inferencia.append(nombre_lag_inferencia)

# Calcula la media movil de los siete dias anteriores sin incluir el dia actual.
media_movil_inferencia = (
    inferencia_df_transformado.set_index("fecha")
    .groupby(["producto_id", "anio"], sort=False)["unidades_vendidas"]
    .rolling("7D", closed="left", min_periods=7)
    .mean()
    .reset_index(level=[0, 1], drop=True)
)
inferencia_df_transformado["media_movil_7_dias"] = (
    media_movil_inferencia.to_numpy()
)

# Quita filas que no tienen historial suficiente para calcular lags y media movil.
filas_inferencia_antes_lags = len(inferencia_df_transformado)
inferencia_df_transformado = inferencia_df_transformado.dropna(
    subset=columnas_lags_inferencia + ["media_movil_7_dias"]
).copy()
filas_inferencia_sin_historial = (
    filas_inferencia_antes_lags - len(inferencia_df_transformado)
)

# Calcula descuento porcentual y resume los precios de los competidores.
inferencia_df_transformado["descuento_porcentaje"] = (
    (
        inferencia_df_transformado["precio_venta"]
        - inferencia_df_transformado["precio_base"]
    )
    / inferencia_df_transformado["precio_base"]
    * 100
)
columnas_precios_competidores_inferencia = [
    "Amazon",
    "Deporvillage",
    "Decathlon",
]
inferencia_df_transformado["precio_competencia"] = (
    inferencia_df_transformado[columnas_precios_competidores_inferencia].mean(
        axis=1
    )
)
inferencia_df_transformado["ratio_precio"] = (
    inferencia_df_transformado["precio_venta"]
    / inferencia_df_transformado["precio_competencia"]
)
inferencia_df_transformado = inferencia_df_transformado.drop(
    columns=columnas_precios_competidores_inferencia
)

# Aplica one-hot encoding a copias de nombre, categoria y subcategoria.
columnas_categoricas_inferencia = ["nombre", "categoria", "subcategoria"]
columnas_categoricas_h_inferencia = [
    f"{columna}_h" for columna in columnas_categoricas_inferencia
]
for columna_inferencia, columna_h_inferencia in zip(
    columnas_categoricas_inferencia,
    columnas_categoricas_h_inferencia,
):
    inferencia_df_transformado[columna_h_inferencia] = (
        inferencia_df_transformado[columna_inferencia].copy()
    )
inferencia_df_transformado = pd.get_dummies(
    inferencia_df_transformado,
    columns=columnas_categoricas_h_inferencia,
    prefix=columnas_categoricas_h_inferencia,
    dtype=np.uint8,
)

# Carga solo el encabezado del CSV de entrenamiento para alinear columnas y orden.
df_esquema_entrenamiento = pd.read_csv(
    "../data/processed/df.csv",
    nrows=0,
)
columnas_esperadas_entrenamiento = df_esquema_entrenamiento.columns.tolist()
columnas_extra_inferencia = [
    columna
    for columna in inferencia_df_transformado.columns
    if columna not in columnas_esperadas_entrenamiento
]
if columnas_extra_inferencia:
    raise ValueError(
        "La inferencia contiene columnas que no existen en entrenamiento: "
        f"{columnas_extra_inferencia}"
    )
inferencia_df_transformado = inferencia_df_transformado.reindex(
    columns=columnas_esperadas_entrenamiento,
    fill_value=0,
)

# Filtra octubre al final; sus filas ya sirvieron para crear lags de noviembre.
filas_antes_filtro_noviembre = len(inferencia_df_transformado)
inferencia_df_transformado = inferencia_df_transformado.loc[
    inferencia_df_transformado["fecha"].dt.month.eq(11)
].copy()
filas_octubre_eliminadas = (
    filas_antes_filtro_noviembre - len(inferencia_df_transformado)
)
assert set(inferencia_df_transformado["fecha"].dt.month.unique()) == {11}
assert inferencia_df_transformado.columns.tolist() == columnas_esperadas_entrenamiento

# Guarda el resultado final con la misma estructura de columnas del entrenamiento.
ruta_inferencia_transformada = (
    "../data/processed/inferencia_df_transformado.csv"
)
inferencia_df_transformado.to_csv(
    ruta_inferencia_transformada,
    index=False,
)

print(f"Filas sin historial suficiente eliminadas: {filas_inferencia_sin_historial:,}")
print(f"Filas no noviembre eliminadas tras transformaciones: {filas_octubre_eliminadas:,}")
print(f"Filas finales de noviembre: {len(inferencia_df_transformado):,}")
print(f"Shape final: {inferencia_df_transformado.shape}")
print("Columnas exactamente alineadas con entrenamiento:", True)
print(f"Archivo guardado: {ruta_inferencia_transformada}")
inferencia_df_transformado[["fecha", "producto_id", "unidades_vendidas"]].head()

Filas sin historial suficiente eliminadas: 864
Filas no noviembre eliminadas tras transformaciones: 0
Filas finales de noviembre: 24
Shape final: (24, 85)
Columnas exactamente alineadas con entrenamiento: True
Archivo guardado: ../data/processed/inferencia_df_transformado.csv


,fecha,producto_id,unidades_vendidas
7,2025-11-01,PROD_001,NaN
44,2025-11-01,PROD_002,NaN
81,2025-11-01,PROD_003,NaN
118,2025-11-01,PROD_004,NaN
155,2025-11-01,PROD_005,NaN
